# 2991. Top Three Wineries

## Problem Description
We need to find the **top three wineries in each country** based on their **total points**.  

- If multiple wineries have the same total points, order them by winery name in ascending order.  
- If there is no second winery, output `"No Second Winery"`.  
- If there is no third winery, output `"No Third Winery"`.  
- Return the result table ordered by `country` in ascending order.  

---

## Schema

### Table: Wineries
| Column Name | Type    | Description                          |
|-------------|---------|--------------------------------------|
| id          | INT     | Unique identifier for each record     |
| country     | VARCHAR | Country of the winery                |
| points      | INT     | Points awarded to the winery         |
| winery      | VARCHAR | Name of the winery                   |

**Primary Key:** `id`

---

## Sample Data

### Wineries
| id  | country   | points | winery          |
|-----|-----------|--------|-----------------|
| 103 | Australia | 84     | WhisperingPines |
| 737 | Australia | 85     | GrapesGalore    |
| 848 | Australia | 100    | HarmonyHill     |
| 222 | Hungary   | 60     | MoonlitCellars  |
| 116 | USA       | 47     | RoyalVines      |
| 124 | USA       | 45     | Eagle'sNest     |
| 648 | India     | 69     | SunsetVines     |
| 894 | USA       | 39     | RoyalVines      |
| 677 | USA       | 9      | PacificCrest    |

---

## Expected Output

| country   | top_winery          | second_winery     | third_winery         |
|-----------|---------------------|-------------------|----------------------|
| Australia | HarmonyHill (100)   | GrapesGalore (85) | WhisperingPines (84) |
| Hungary   | MoonlitCellars (60) | No Second Winery  | No Third Winery      |
| India     | SunsetVines (69)    | No Second Winery  | No Third Winery      |
| USA       | RoyalVines (86)     | Eagle'sNest (45)  | PacificCrest (9)     |

---

## Explanation
- **Australia**:  
  - HarmonyHill → 100 points (highest)  
  - GrapesGalore → 85 points (second)  
  - WhisperingPines → 84 points (third)  

- **Hungary**:  
  - MoonlitCellars → 60 points (only winery)  
  - No second or third winery  

- **India**:  
  - SunsetVines → 69 points (only winery)  
  - No second or third winery  

- **USA**:  
  - RoyalVines → 47 + 39 = 86 points (highest)  
  - Eagle'sNest → 45 points (second)  
  - PacificCrest → 9 points (third)  

---

## PySpark Code: Create DataFrame and Temp View

```python


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from datetime import date

# Schema for Wineries
wineries_schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("country", StringType(), False),
    StructField("points", IntegerType(), False),
    StructField("winery", StringType(), False)
])

# Data for Wineries
wineries_data = [
    (103, "Australia", 84, "WhisperingPines"),
    (737, "Australia", 85, "GrapesGalore"),
    (848, "Australia", 100, "HarmonyHill"),
    (222, "Hungary", 60, "MoonlitCellars"),
    (116, "USA", 47, "RoyalVines"),
    (124, "USA", 45, "Eagle'sNest"),
    (648, "India", 69, "SunsetVines"),
    (894, "USA", 39, "RoyalVines"),
    (677, "USA", 9, "PacificCrest")
]

# Create DataFrame
wineries_df = spark.createDataFrame(wineries_data, wineries_schema)

# Register Temp View
wineries_df.createOrReplaceTempView("Wineries")

# Quick check
wineries_df.show()


In [0]:
%sql
with cte as (
    select concat(winery , ' (' ,points,')') as winery_cte ,
  dense_rank()over(partition by country  order by  points desc , winery asc ) as rnk,
    *

    from Wineries
)
Select 
t1.country  
,t1.winery_cte as top_winer 
,
coalesce(t2.winery_cte  , 'No second winery') as second_winer,
coalesce(t3.winery_cte,'No third winery') as third_winer

from cte as t1 
left join cte as t2  on t1.country = t2.country and t1.rnk = t2.rnk -1
left join cte as t3 on t1.country = t3.country and t2.rnk = t3.rnk -1
where t1.rnk =1 

In [0]:
%sql
with cte as (
    Select concat(winery , ' (' ,points,')') as winery_cte ,
  rank()over(partition by country  order by  points desc , winery asc ) as rnk,
  * from Wineries
)
Select 
country , 
coalesce(max(case when rnk =1 then winery_cte end), 'No first winery') as top_winer ,
coalesce(max(case when rnk =2 then winery_cte end), 'No second winery') as second_winer,
coalesce(max(case when rnk =3 then winery_cte end), 'No third winery') as third_winer
 from cte 
 group by country
 order by country asc